In [1]:
import tensorflow as tf
from tensorflow.keras import backend as K
import pickle
import numpy as np
import pandas as pd
import os
import json
import sys
import warnings
from pathlib import Path
from pprint import pformat
from typing import Dict, Union
from create_data_generator import data_generator, batch_predict

2024-11-14 08:23:59.209166: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-14 08:24:00.022526: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# load models for preprocessed data
cancer_gen_expr_model = tf.keras.models.load_model(os.path.join('exp_result',"cancer_gen_expr_model"))
cancer_gen_mut_model = tf.keras.models.load_model(os.path.join('exp_result', "cancer_gen_mut_model"))
cancer_dna_methy_model = tf.keras.models.load_model(os.path.join('exp_result', "cancer_dna_methy_model"))

2024-11-14 08:24:12.318690: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-14 08:24:14.394268: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 30960 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:86:00.0, compute capability: 7.0


In [3]:
cancer_gen_expr_model.trainable = False
cancer_gen_mut_model.trainable = False
cancer_dna_methy_model.trainable = False

In [4]:
with open(os.path.join('exp_result', "drug_features.pickle"),"rb") as f:
    dict_features = pickle.load(f)

with open(os.path.join('exp_result', "norm_adj_mat.pickle"),"rb") as f:
    dict_adj_mat = pickle.load(f)

In [5]:
test_keep = pd.read_csv(os.path.join('exp_result', "test_y_data.csv"))
test_keep.columns = ["Cell_Line", "Drug_ID", "AUC"]

In [6]:
%%time
test_gcn_feats = []
test_adj_list = []
for drug_id in test_keep["Drug_ID"].values:
    test_gcn_feats.append(dict_features[drug_id])
    test_adj_list.append(dict_adj_mat[drug_id])

CPU times: user 942 µs, sys: 0 ns, total: 942 µs
Wall time: 5 ms


In [7]:
%%time
test_gcn_feats = np.array(test_gcn_feats).astype("float32")
test_adj_list = np.array(test_adj_list).astype("float32")

CPU times: user 60.4 ms, sys: 84.6 ms, total: 145 ms
Wall time: 145 ms


In [8]:
# load trained model here
CCLE_model = tf.keras.models.load_model('exp_result/DeepCDR_model/DeepCDR_model')

In [9]:
CCLE_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 223, 223)]   0           []                               
                                                                                                  
 input_1 (InputLayer)           [(None, 223, 75)]    0           []                               
                                                                                                  
 dot (Dot)                      (None, 223, 75)      0           ['input_2[0][0]',                
                                                                  'input_1[0][0]']                
                                                                                                  
 dense (Dense)                  (None, 223, 256)     19456       ['dot[0][0]']                

In [10]:
samp_drug = test_keep["Drug_ID"].unique()[-1]
samp_ach = np.array(test_keep["Cell_Line"].unique()[-1])

In [11]:
samp_drug, samp_ach

('Drug_665', array('ACH-000828', dtype='<U10'))

In [12]:
# redefine the model
training = True
dropout1 = 0.10
dropout2 = 0.20

input_gcn_features = tf.keras.layers.Input(shape = (dict_features[samp_drug].shape[0], 75))
input_norm_adj_mat = tf.keras.layers.Input(shape = (dict_adj_mat[samp_drug].shape[0], dict_adj_mat[samp_drug].shape[0]))
mult_1 = tf.keras.layers.Dot(1)([input_norm_adj_mat, input_gcn_features])
dense_layer_gcn = tf.keras.layers.Dense(256, activation = "relu")
dense_out = dense_layer_gcn(mult_1)
dense_out = tf.keras.layers.BatchNormalization()(dense_out)
dense_out = tf.keras.layers.Dropout(dropout1)(dense_out, training = training)
mult_2 = tf.keras.layers.Dot(1)([input_norm_adj_mat, dense_out])
dense_layer_gcn = tf.keras.layers.Dense(256, activation = "relu")
dense_out = dense_layer_gcn(mult_2)
dense_out = tf.keras.layers.BatchNormalization()(dense_out)
dense_out = tf.keras.layers.Dropout(dropout1)(dense_out, training = training)

dense_layer_gcn = tf.keras.layers.Dense(100, activation = "relu")
mult_3 = tf.keras.layers.Dot(1)([input_norm_adj_mat, dense_out])
dense_out = dense_layer_gcn(mult_3)
dense_out = tf.keras.layers.BatchNormalization()(dense_out)
dense_out = tf.keras.layers.Dropout(dropout1)(dense_out, training = training)

dense_out = tf.keras.layers.GlobalAvgPool1D()(dense_out)
# All above code is for GCN for drugs

# methylation data
input_gen_methy1 = tf.keras.layers.Input(shape = (1,), dtype = tf.string)
input_gen_methy = cancer_dna_methy_model(input_gen_methy1)
input_gen_methy.trainable = False
gen_methy_layer = tf.keras.layers.Dense(256, activation = "tanh")
    
gen_methy_emb = gen_methy_layer(input_gen_methy)
gen_methy_emb = tf.keras.layers.BatchNormalization()(gen_methy_emb)
gen_methy_emb = tf.keras.layers.Dropout(dropout1)(gen_methy_emb, training = training)
gen_methy_layer = tf.keras.layers.Dense(100, activation = "relu")
gen_methy_emb = gen_methy_layer(gen_methy_emb)

# gene expression data
input_gen_expr1 = tf.keras.layers.Input(shape = (1,), dtype = tf.string)
input_gen_expr = cancer_gen_expr_model(input_gen_expr1)
input_gen_expr.trainable = False
gen_expr_layer = tf.keras.layers.Dense(256, activation = "tanh")
    
gen_expr_emb = gen_expr_layer(input_gen_expr)
gen_expr_emb = tf.keras.layers.BatchNormalization()(gen_expr_emb)
gen_expr_emb = tf.keras.layers.Dropout(dropout1)(gen_expr_emb, training = training)
gen_expr_layer = tf.keras.layers.Dense(100, activation = "relu")
gen_expr_emb = gen_expr_layer(gen_expr_emb)
    
    
input_gen_mut1 = tf.keras.layers.Input(shape = (1,), dtype = tf.string)
input_gen_mut = cancer_gen_mut_model(input_gen_mut1)
input_gen_mut.trainable = False
    
reshape_gen_mut = tf.keras.layers.Reshape((1, cancer_gen_mut_model(samp_ach).numpy().shape[0], 1))
reshape_gen_mut = reshape_gen_mut(input_gen_mut)
gen_mut_layer = tf.keras.layers.Conv2D(50, (1, 700), strides=5, activation = "tanh")
gen_mut_emb = gen_mut_layer(reshape_gen_mut)
pool_layer = tf.keras.layers.MaxPooling2D((1,5))
pool_out = pool_layer(gen_mut_emb)
gen_mut_layer = tf.keras.layers.Conv2D(30, (1, 5), strides=2, activation = "relu")
gen_mut_emb = gen_mut_layer(pool_out)
pool_layer = tf.keras.layers.MaxPooling2D((1,10))
pool_out = pool_layer(gen_mut_emb)
flatten_layer = tf.keras.layers.Flatten()
flatten_out = flatten_layer(pool_out)
x_mut = tf.keras.layers.Dense(100,activation = 'relu')(flatten_out)
x_mut = tf.keras.layers.Dropout(dropout1)(x_mut, training = training)
    
all_omics = tf.keras.layers.Concatenate()([dense_out, gen_methy_emb, gen_expr_emb, x_mut])
x = tf.keras.layers.Dense(300,activation = 'tanh')(all_omics)
x = tf.keras.layers.Dropout(dropout1)(x, training = training)
x = tf.keras.layers.Lambda(lambda x: K.expand_dims(x,axis=-1))(x)
x = tf.keras.layers.Lambda(lambda x: K.expand_dims(x,axis=1))(x)
x = tf.keras.layers.Conv2D(filters=30, kernel_size=(1,150),strides=(1, 1), activation = 'relu',padding='valid')(x)
x = tf.keras.layers.MaxPooling2D(pool_size=(1,2))(x)
x = tf.keras.layers.Conv2D(filters=10, kernel_size=(1,5),strides=(1, 1), activation = 'relu',padding='valid')(x)
x = tf.keras.layers.MaxPooling2D(pool_size=(1,3))(x)
x = tf.keras.layers.Conv2D(filters=5, kernel_size=(1,5),strides=(1, 1), activation = 'relu',padding='valid')(x)
x = tf.keras.layers.MaxPooling2D(pool_size=(1,3))(x)
x = tf.keras.layers.Dropout(dropout1)(x, training = training)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dropout(dropout2)(x, training = training)
final_out_layer = tf.keras.layers.Dense(1, activation = "linear")
final_out = final_out_layer(x)
simplecdr = tf.keras.models.Model([input_gcn_features, input_norm_adj_mat, input_gen_expr1,
                                   input_gen_methy1, input_gen_mut1], final_out)

In [13]:
simplecdr.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 223, 223)]   0           []                               
                                                                                                  
 input_1 (InputLayer)           [(None, 223, 75)]    0           []                               
                                                                                                  
 dot (Dot)                      (None, 223, 75)      0           ['input_2[0][0]',                
                                                                  'input_1[0][0]']                
                                                                                                  
 dense (Dense)                  (None, 223, 256)     19456       ['dot[0][0]']                

In [14]:
# do get and set weights?


In [15]:
weights_train = CCLE_model.get_weights()

In [16]:
simplecdr.set_weights(weights_train)

In [17]:
weights_new = simplecdr.get_weights()

In [18]:
print(len(weights_train), len(weights_new))

53 53


In [19]:
# for i in range(len(weights_train)):
#     print(np.mean(weights_train[i] == weights_new[i]))

In [20]:
# Weights match, I think we just do the predictions now

In [21]:
# # get the predictions on the test set
generator_batch_size = 32
test_steps = int(np.ceil(len(test_gcn_feats) / generator_batch_size))
preds_test, target_test = batch_predict(simplecdr, data_generator(test_gcn_feats, test_adj_list, test_keep["Cell_Line"].values.reshape(-1,1), test_keep["Cell_Line"].values.reshape(-1,1), test_keep["Cell_Line"].values.reshape(-1,1), test_keep["AUC"].values.reshape(-1,1), generator_batch_size, shuffle = False), test_steps)   

Generating batch from index 0 to 32


2024-11-14 08:24:21.277394: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inmodel/dropout_7/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2024-11-14 08:24:22.976365: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:428] Loaded cuDNN version 8401


Generating batch from index 32 to 64
Generating batch from index 64 to 96
Generating batch from index 96 to 128
Generating batch from index 128 to 160
Generating batch from index 160 to 192
Generating batch from index 192 to 224
Generating batch from index 224 to 256
Generating batch from index 256 to 288
Generating batch from index 288 to 320
Generating batch from index 320 to 352
Generating batch from index 352 to 384
Generating batch from index 384 to 416
Generating batch from index 416 to 448
Generating batch from index 448 to 480
Generating batch from index 480 to 512
Generating batch from index 512 to 544
Generating batch from index 544 to 576
Generating batch from index 576 to 608
Generating batch from index 608 to 640
Generating batch from index 640 to 672
Generating batch from index 672 to 704
Generating batch from index 704 to 736
Generating batch from index 736 to 768
Generating batch from index 768 to 800
Generating batch from index 800 to 832
Generating batch from index 83

2024-11-14 08:24:27.111317: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inmodel/dropout_7/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


In [22]:
print(preds_test.shape, target_test.shape)

(951,) (951,)


In [23]:
# Write a for loop for this?

In [24]:
%%time
all_predicted_values = []
for i in range(25):
    preds_test, target_test = batch_predict(simplecdr, data_generator(test_gcn_feats, test_adj_list, test_keep["Cell_Line"].values.reshape(-1,1), test_keep["Cell_Line"].values.reshape(-1,1), test_keep["Cell_Line"].values.reshape(-1,1), test_keep["AUC"].values.reshape(-1,1), generator_batch_size, shuffle = False), test_steps)
    all_predicted_values.append(preds_test)

Generating batch from index 0 to 32
Generating batch from index 32 to 64
Generating batch from index 64 to 96
Generating batch from index 96 to 128
Generating batch from index 128 to 160
Generating batch from index 160 to 192
Generating batch from index 192 to 224
Generating batch from index 224 to 256
Generating batch from index 256 to 288
Generating batch from index 288 to 320
Generating batch from index 320 to 352
Generating batch from index 352 to 384
Generating batch from index 384 to 416
Generating batch from index 416 to 448
Generating batch from index 448 to 480
Generating batch from index 480 to 512
Generating batch from index 512 to 544
Generating batch from index 544 to 576
Generating batch from index 576 to 608
Generating batch from index 608 to 640
Generating batch from index 640 to 672
Generating batch from index 672 to 704
Generating batch from index 704 to 736
Generating batch from index 736 to 768
Generating batch from index 768 to 800
Generating batch from index 800 t

In [25]:
# preds_test

In [26]:
# target_test

In [27]:
preds_df = pd.DataFrame(all_predicted_values)

In [28]:
preds_df_final = preds_df.T

In [29]:
preds_df_final.head()

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,0.851870,1.115308,1.598304,1.092197,1.323720,1.202901,0.716995,0.863523,0.377259,1.154572,...,0.795938,0.574391,1.381686,0.071204,-0.096443,0.922886,1.433315,0.910785,0.619118,1.125842
1,0.995986,0.443633,0.523349,1.472106,1.403751,0.151907,1.366053,1.007318,0.557388,1.373289,...,0.561323,0.715496,0.422111,0.743618,0.397674,1.206003,0.568994,0.686288,0.847013,1.395890
2,0.340300,0.407578,0.392172,0.304199,0.749364,0.368177,0.769473,0.033115,0.246899,0.209982,...,0.099552,0.565988,1.216202,0.298496,-0.051851,0.542991,0.792377,0.343482,0.323088,0.243969
3,0.746661,0.897285,0.493810,0.212299,0.733260,0.725582,0.778590,0.319857,0.729852,0.434241,...,0.300684,0.862187,0.969051,0.867353,0.843785,0.359850,0.349319,0.301470,0.983724,0.638810
4,0.512972,1.293912,1.241898,0.808111,1.533287,0.300312,0.989831,1.420643,1.089652,0.923130,...,1.433160,1.501443,1.194283,0.810319,1.123132,0.674565,0.630459,0.002293,0.307540,1.458885


In [30]:
preds_df_final.shape

(951, 25)

In [31]:
preds_df_final.columns = ['predicted_vals_' + str(i + 1) for i in range(preds_df_final.shape[1])]

In [32]:
preds_df_final.head()

,predicted_vals_1,predicted_vals_2,predicted_vals_3,predicted_vals_4,predicted_vals_5,predicted_vals_6,predicted_vals_7,predicted_vals_8,predicted_vals_9,predicted_vals_10,...,predicted_vals_16,predicted_vals_17,predicted_vals_18,predicted_vals_19,predicted_vals_20,predicted_vals_21,predicted_vals_22,predicted_vals_23,predicted_vals_24,predicted_vals_25
0,0.851870,1.115308,1.598304,1.092197,1.323720,1.202901,0.716995,0.863523,0.377259,1.154572,...,0.795938,0.574391,1.381686,0.071204,-0.096443,0.922886,1.433315,0.910785,0.619118,1.125842
1,0.995986,0.443633,0.523349,1.472106,1.403751,0.151907,1.366053,1.007318,0.557388,1.373289,...,0.561323,0.715496,0.422111,0.743618,0.397674,1.206003,0.568994,0.686288,0.847013,1.395890
2,0.340300,0.407578,0.392172,0.304199,0.749364,0.368177,0.769473,0.033115,0.246899,0.209982,...,0.099552,0.565988,1.216202,0.298496,-0.051851,0.542991,0.792377,0.343482,0.323088,0.243969
3,0.746661,0.897285,0.493810,0.212299,0.733260,0.725582,0.778590,0.319857,0.729852,0.434241,...,0.300684,0.862187,0.969051,0.867353,0.843785,0.359850,0.349319,0.301470,0.983724,0.638810
4,0.512972,1.293912,1.241898,0.808111,1.533287,0.300312,0.989831,1.420643,1.089652,0.923130,...,1.433160,1.501443,1.194283,0.810319,1.123132,0.674565,0.630459,0.002293,0.307540,1.458885


In [33]:
# join the target counts here

In [34]:
combined_df = pd.concat((preds_df_final, pd.DataFrame(target_test, columns = ['target_auc'])), axis = 1)

In [35]:
combined_df.head()

,predicted_vals_1,predicted_vals_2,predicted_vals_3,predicted_vals_4,predicted_vals_5,predicted_vals_6,predicted_vals_7,predicted_vals_8,predicted_vals_9,predicted_vals_10,...,predicted_vals_17,predicted_vals_18,predicted_vals_19,predicted_vals_20,predicted_vals_21,predicted_vals_22,predicted_vals_23,predicted_vals_24,predicted_vals_25,target_auc
0,0.851870,1.115308,1.598304,1.092197,1.323720,1.202901,0.716995,0.863523,0.377259,1.154572,...,0.574391,1.381686,0.071204,-0.096443,0.922886,1.433315,0.910785,0.619118,1.125842,0.8148
1,0.995986,0.443633,0.523349,1.472106,1.403751,0.151907,1.366053,1.007318,0.557388,1.373289,...,0.715496,0.422111,0.743618,0.397674,1.206003,0.568994,0.686288,0.847013,1.395890,0.8126
2,0.340300,0.407578,0.392172,0.304199,0.749364,0.368177,0.769473,0.033115,0.246899,0.209982,...,0.565988,1.216202,0.298496,-0.051851,0.542991,0.792377,0.343482,0.323088,0.243969,0.4573
3,0.746661,0.897285,0.493810,0.212299,0.733260,0.725582,0.778590,0.319857,0.729852,0.434241,...,0.862187,0.969051,0.867353,0.843785,0.359850,0.349319,0.301470,0.983724,0.638810,0.4997
4,0.512972,1.293912,1.241898,0.808111,1.533287,0.300312,0.989831,1.420643,1.089652,0.923130,...,1.501443,1.194283,0.810319,1.123132,0.674565,0.630459,0.002293,0.307540,1.458885,0.7675


In [36]:
preds_df_final.values

array([[ 0.8518704 ,  1.1153079 ,  1.5983038 , ...,  0.9107855 ,
         0.61911815,  1.1258425 ],
       [ 0.99598634,  0.44363254,  0.52334946, ...,  0.6862883 ,
         0.84701306,  1.3958895 ],
       [ 0.34029973,  0.40757847,  0.3921724 , ...,  0.34348226,
         0.3230878 ,  0.2439686 ],
       ...,
       [ 1.0235237 ,  0.07574624,  1.253286  , ..., -0.03218883,
         0.46125156,  0.73409045],
       [ 1.1129298 ,  0.68628466,  0.8201932 , ...,  1.112372  ,
         1.3796585 ,  1.0573535 ],
       [ 1.3169699 ,  1.2295016 ,  1.2705445 , ...,  1.0573876 ,
         0.85631585,  0.93121415]], dtype=float32)

In [37]:
# proceed to computing the widths and the coverages

In [38]:
li_test = np.percentile(preds_df_final, axis = 1, q = (2.5, 97.5))[0,:].reshape(-1,1)     
ui_test = np.percentile(preds_df_final, axis = 1, q = (2.5, 97.5))[1,:].reshape(-1,1)   

In [39]:
width_test = ui_test - li_test

In [40]:
avg_width_test = width_test.mean(0)[0]
avg_width_test

1.3171484025287294

In [41]:
ind_test = (target_test >= li_test) & (target_test <= ui_test)
coverage_test= ind_test.mean(0)[0]
coverage_test

0.9947423764458465